## Install Required Libraries & Fetch Code from repo

In [1]:
pip install sacrebleu rouge_score transformers langgraph langchain chromadb langchain rapidfuzz langchain_openai python-dotenv typing ragas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the depe

## Setup

In [ ]:
HF_TOKEN=""

**GENERATOR**

In [ ]:
from transformers import pipeline

run = pipeline("text-generation", model="google/gemma-3-1b-it", token = HF_TOKEN)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda:0


**Embedding Gemma**

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("google/embeddinggemma-300m", token = HF_TOKEN)

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

**LOCAL VECTORDB**

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="RAG_Evaluation")

**TEXT SPLITTERS FOR CHUNKING**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)

## RAG Setup

In [31]:
from typing import TypedDict

class State(TypedDict):
  query: str
  context: list[str]
  answer: str

**NODES**

In [32]:
def retreive(state:State):
  """
  query RAG using a local vector database
  """

  context = collection.query(
    query_embeddings = embedder.encode(state['query']),
    n_results = 10,
    include = ["documents", "distances"]
  )['documents']

  return {"context":context}

In [33]:
def upsert(texts, num:int):
  embeds = embedder.encode(texts)
  collection.upsert(
      embeddings = embeds,
      ids= [f"id{num + i}" for i in range(len(texts))],
      documents = texts
  )

In [34]:
from langchain_core.prompts import ChatPromptTemplate

answering_prompt = """
  You are an assistant that answers the user's question using ONLY the provided context.
  if you don't know the answer say - i don't know the answer
"""

def answer(state:State):
  """
  generate the final answer using the LLM and retrieved documents
  """

  prompt = ChatPromptTemplate([
    ("system", answering_prompt),
    ("user", "Question: {question} \n\n Context: {context}"),
  ])

  query = prompt.invoke({"question":state['query'], "context":state['context']}).messages

  # Convert for gemma model
  queries = []
  for q in query:
      role = 'user' if q.type == 'human' else q.type

      queries.append({
          'role': role,
          'content': q.content
      })
  result = run(queries)

  return {"answer":result[0]['generated_text'][-1]['content']}


**RAG**

In [35]:
from langgraph.graph import StateGraph, START, END

rag = (
    StateGraph(State)
    .add_sequence([retreive, answer])
    .add_edge(START, "retreive")
    .compile()
)

In [36]:
rag.invoke({"query":"what does this texts means ?"})

{'query': 'what does this texts means ?',
 'context': [[]],
 'answer': "i don't know the answer\n"}

## Perpare Evaluation Data

**Upsert all docs in local vectordb**

In [37]:
import json

papers = {}

splited_num = 0
with open("qasper_papers.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        paper = json.loads(line)
        print(f"------------------------------ UPSERTED [{i}][148]")

        splited_text = text_splitter.split_text(paper[list(paper)[0]])

        upsert(splited_text, splited_num)
        splited_num += len(splited_text)

------------------------------ UPSERTED [11][148]
------------------------------ UPSERTED [12][148]
------------------------------ UPSERTED [13][148]
------------------------------ UPSERTED [14][148]
------------------------------ UPSERTED [15][148]
------------------------------ UPSERTED [16][148]
------------------------------ UPSERTED [17][148]
------------------------------ UPSERTED [18][148]
------------------------------ UPSERTED [19][148]
------------------------------ UPSERTED [20][148]
------------------------------ UPSERTED [21][148]
------------------------------ UPSERTED [22][148]
------------------------------ UPSERTED [23][148]
------------------------------ UPSERTED [24][148]
------------------------------ UPSERTED [25][148]
------------------------------ UPSERTED [26][148]
------------------------------ UPSERTED [27][148]
------------------------------ UPSERTED [28][148]
------------------------------ UPSERTED [29][148]
------------------------------ UPSERTED [30][148]


**Generate All Answers**

In [38]:
import json
Questions = []

with open("qasper_qa(1).jsonl", "r", encoding="utf-8") as f:
    for line in f:
        qa = json.loads(line)
        Questions.append(qa)

In [40]:
for i in range(len(Questions)):
  Questions[i]['outputs'] = rag.invoke({"query":Questions[i]['input']})

  if i%10 == 0:
    print(f"|-------------------------------- [{i}]")

|-------------------------------- [0]


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


|-------------------------------- [10]
|-------------------------------- [20]
|-------------------------------- [30]
|-------------------------------- [40]
|-------------------------------- [50]
|-------------------------------- [60]
|-------------------------------- [70]
|-------------------------------- [80]
|-------------------------------- [90]
|-------------------------------- [100]
|-------------------------------- [110]
|-------------------------------- [120]


In [43]:
with open("qasper_mini_qa_generated.jsonl", "w") as f:
    for row in Questions:
        f.write(json.dumps(row) + "\n")